# Baseline classiche per il denoising di immagini con rumore Poisson

In questo notebook vengono valutati alcuni metodi classici di denoising su immagini degradate da **rumore Poisson**.

L'obiettivo è costruire un confronto sperimentale con i risultati ottenuti tramite **Noise2Void**, utilizzando gli stessi dataset e le stesse metriche di valutazione.

Il rumore Poisson è particolarmente rilevante nell'elaborazione delle immagini perché può modellare fenomeni legati al processo di acquisizione, come il rumore di conteggio dei fotoni. A differenza del rumore gaussiano additivo, il rumore Poisson è **dipendente dal segnale**, poiché la sua varianza dipende dall'intensità del pixel.

## Obiettivo del notebook

In questo esperimento vengono applicati e confrontati diversi metodi di denoising:

- filtro media;
- filtro mediano;
- trasformazione di Anscombe combinata con BM3D.

Il filtro media e il filtro mediano vengono utilizzati come baseline semplici.  
La combinazione **Anscombe + BM3D** viene invece utilizzata come baseline più specifica per il rumore Poisson.

## Perché usare la trasformazione di Anscombe

Il rumore Poisson è un rumore signal-dependent, cioè la sua intensità dipende dal valore del segnale osservato.  
Per questo motivo, prima di applicare metodi pensati per il rumore gaussiano, è comune utilizzare una trasformazione di stabilizzazione della varianza.

La trasformazione di Anscombe è definita come:

```text
A(x) = 2 * sqrt(x + 3/8)

## Installazione della libreria BM3D

BM3D è un algoritmo classico di denoising particolarmente efficace per il rumore gaussiano.  
Nel notebook viene utilizzata una libreria Python già disponibile tramite `pip`.
In questo caso BM3D sarà applicato alla trasformata di Anscombe.

In [2]:
!pip install -q bm3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.0/862.0 kB 23.2 MB/s eta 0:00:0000:01


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import cv2
import bm3d


from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

## Definizione dei percorsi dei dataset

In [5]:
# -----------------------------
# CLEAN DATASET
# -----------------------------

kodak_clean_dir = Path(
    "/kaggle/input/datasets/sherylmehta/kodak-dataset"
)

bsd500_clean_dir = Path(
    "/kaggle/input/datasets/balraj98/berkeley-segmentation-dataset-500-bsds500/images"
)

div2k_clean_base_dir = Path(
    "/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images"
)


# -----------------------------
# NOISY POISSON DATASET
# -----------------------------

kodak_noisy_dir = Path(
    "/kaggle/input/notebooks/antoiann/poissonnoise/kodak_poisson_lambda30"
)

bsd500_noisy_dir = Path(
    "/kaggle/input/notebooks/antoiann/poissonnoise/bsd500_poisson_lambda10"
)

div2k_noisy_base_dir = Path(
    "/kaggle/input/notebooks/antoiann/poissonnoise/div2k_poisson_lambda10"
)


# -----------------------------
# OUTPUT BASELINES
# -----------------------------

output_base_dir = Path("/kaggle/working/poisson_classical_baselines")
output_base_dir.mkdir(parents=True, exist_ok=True)

## Definizione degli split dei dataset

BSD500 è già suddiviso in `train`, `val` e `test`.  
Per la valutazione finale delle baseline viene utilizzato principalmente lo split `test`.

DIV2K, invece, è organizzato nelle cartelle `DIV2K_train_HR` e `DIV2K_valid_HR`.  
Per la valutazione finale viene usato `DIV2K_valid_HR`.

Kodak non presenta una suddivisione interna e viene usato interamente come dataset di test esterno.

In [7]:
# -----------------------------
# BSD500 split
# -----------------------------

bsd500_train_clean_dir = bsd500_clean_dir / "train"
bsd500_val_clean_dir   = bsd500_clean_dir / "val"
bsd500_test_clean_dir  = bsd500_clean_dir / "test"

bsd500_train_noisy_dir = bsd500_noisy_dir / "train"
bsd500_val_noisy_dir   = bsd500_noisy_dir / "val"
bsd500_test_noisy_dir  = bsd500_noisy_dir / "test"


# -----------------------------
# DIV2K split
# -----------------------------

div2k_train_clean_dir = div2k_clean_base_dir / "DIV2K_train_HR" / "DIV2K_train_HR"
div2k_valid_clean_dir = div2k_clean_base_dir / "DIV2K_valid_HR" / "DIV2K_valid_HR"

div2k_train_noisy_dir = div2k_noisy_base_dir / "DIV2K_train_HR" / "DIV2K_train_HR"
div2k_valid_noisy_dir = div2k_noisy_base_dir / "DIV2K_valid_HR" / "DIV2K_valid_HR"

## Definizione dei metodi di denoising classici

In questa sezione vengono definite le funzioni per applicare i metodi classici di denoising.

I metodi considerati sono:

- filtro media;
- filtro mediano;
- Anscombe+BM3D.

Le immagini vengono gestite come array `float32` normalizzati nell'intervallo `[0, 1]`.  
Quando necessario, vengono temporaneamente convertite in `uint8` per l'utilizzo con OpenCV.

## Trasformazione di Anscombe e BM3D

Nel caso del rumore Poisson, la varianza del rumore dipende dall'intensità del segnale.  
Questo significa che il rumore non ha la stessa intensità in tutte le zone dell'immagine: le regioni più luminose e quelle più scure possono essere degradate in modo diverso.

Per applicare metodi pensati principalmente per rumore gaussiano, come BM3D, è comune usare prima una trasformazione di stabilizzazione della varianza.

### Trasformazione di Anscombe

La trasformazione di Anscombe è definita come:

```text
A(x) = 2 * sqrt(x + 3/8) 
```
### Trasformazione inversa

Dopo aver applicato il denoising nello spazio trasformato, è necessario riportare l'immagine nello spazio originale.
Per questo motivo viene utilizzata una trasformazione inversa approssimata:
```

A^(-1)(y) = (y / 2)^2 - 3/8
```

### immagine rumorosa Poisson
        ↓
### trasformazione di Anscombe
        ↓
### BM3D nello spazio trasformato
        ↓
### trasformazione inversa di Anscombe
        ↓
### immagine denoised finale

In [8]:
def anscombe_transform(x):
    """
    Applica la trasformazione di Anscombe.

    x: immagine float non negativa.
    """
# Serve a stabilizzare la varianza del rumore Poisson,
# rendendolo più simile a un rumore gaussiano.
    x = np.clip(x, 0, None)
    return 2.0 * np.sqrt(x + 3.0 / 8.0)


def inverse_anscombe_transform(y):
    """
    Applica l'inversa approssimata della trasformazione di Anscombe.
    """
# Riporta l'immagine denoised dallo spazio trasformato
# allo spazio originale.
    restored = (y / 2.0) ** 2 - 3.0 / 8.0
    return np.clip(restored, 0, None)

In [9]:
def apply_anscombe_bm3d_rgb(noisy_np, sigma=1.0):
    """
    Applica Anscombe + BM3D + inversa di Anscombe a un'immagine RGB.

    noisy_np: immagine float in [0,1]
    sigma: deviazione standard approssimata nello spazio Anscombe.
    """
# Per ogni canale RGB:
# 1. applica Anscombe;
# 2. applica BM3D nello spazio trasformato;
# 3. applica l'inversa di Anscombe;
# 4. ricompone l'immagine finale.
    denoised = np.zeros_like(noisy_np, dtype=np.float32)

    for c in range(3):
        channel = noisy_np[:, :, c]

        # Anscombe transform
        transformed = anscombe_transform(channel)

        # BM3D nello spazio trasformato
        den_transformed = bm3d.bm3d(
            transformed,
            sigma_psd=sigma
        )

        # Inversa Anscombe
        restored = inverse_anscombe_transform(den_transformed)

        denoised[:, :, c] = restored

    denoised = np.clip(denoised, 0, 1)

    return denoised

### Filtro media

Il filtro media sostituisce ogni pixel con la media dei valori presenti in una finestra locale.  
È un filtro semplice e può ridurre il rumore gaussiano, ma tende anche a sfocare bordi e dettagli fini dell'immagine.

In [10]:
def apply_mean_filter(noisy_np, kernel_size=3):
    """
    Applica un filtro media a un'immagine RGB float in [0,1].
    """
    # Conversione da float [0,1] a uint8 [0,255]
    noisy_uint8 = (np.clip(noisy_np, 0, 1) * 255).astype(np.uint8)
    # Applicazione del filtro media tramite OpenCV
    filtered_uint8 = cv2.blur(
        noisy_uint8,
        (kernel_size, kernel_size)
    )

    filtered_np = filtered_uint8.astype(np.float32) / 255.0

    return filtered_np

### Filtro mediano

Il filtro mediano sostituisce ogni pixel con la mediana dei valori presenti in una finestra locale.  
È particolarmente efficace contro rumori impulsivi, come salt-and-pepper, ma viene valutato anche sul rumore gaussiano come baseline classica semplice.

In [11]:
def apply_median_filter(noisy_np, kernel_size=3):
    """
    Applica un filtro mediano a un'immagine RGB float in [0,1].
    """

    noisy_uint8 = (np.clip(noisy_np, 0, 1) * 255).astype(np.uint8)

    filtered_uint8 = cv2.medianBlur(
        noisy_uint8,
        kernel_size
    )

    filtered_np = filtered_uint8.astype(np.float32) / 255.0

    return filtered_np

## Funzione generale di valutazione

La funzione `evaluate_baseline_dataset` applica un metodo di denoising a tutte le immagini rumorose di un dataset, salva le immagini denoised e calcola le metriche quantitative rispetto alle immagini pulite.

Per ogni immagine vengono calcolati:

- PSNR clean/noisy;
- SSIM clean/noisy;
- PSNR clean/denoised;
- SSIM clean/denoised;
- gain PSNR;
- gain SSIM.

I risultati vengono salvati in un file CSV, così da poter essere analizzati e confrontati successivamente.

In [12]:
def evaluate_baseline_dataset(
    method_name,
    filter_function,
    noisy_dir,
    clean_dir,
    output_dir,
    csv_name="metrics.csv",
    max_images=None
):
    """
    Applica un metodo di denoising classico a tutte le immagini rumorose
    contenute in noisy_dir, salva gli output e calcola PSNR/SSIM rispetto
    alle immagini clean corrispondenti.
    """

    noisy_dir = Path(noisy_dir)
    clean_dir = Path(clean_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    noisy_paths = get_image_paths(noisy_dir)

    if max_images is not None:
        noisy_paths = noisy_paths[:max_images]

    results = []

    for noisy_path in tqdm(noisy_paths, desc=method_name):
        try:
            relative_path = noisy_path.relative_to(noisy_dir)
            clean_path = clean_dir / relative_path

            if not clean_path.exists():
                print("Clean non trovata:", clean_path)
                continue

            # Lettura noisy
            noisy_img = Image.open(noisy_path).convert("RGB")
            noisy_np = np.array(noisy_img).astype(np.float32) / 255.0

            # Lettura clean
            clean_img = Image.open(clean_path).convert("RGB")
            clean_img = clean_img.resize((noisy_np.shape[1], noisy_np.shape[0]))
            clean_np = np.array(clean_img).astype(np.float32) / 255.0

            # Applicazione filtro
            denoised_np = filter_function(noisy_np)
            denoised_np = np.clip(denoised_np, 0, 1)

            # Salvataggio immagine denoised
            save_path = output_dir / relative_path
            save_path = save_path.with_suffix(".png")
            save_path.parent.mkdir(parents=True, exist_ok=True)

            Image.fromarray(
                (denoised_np * 255).astype(np.uint8)
            ).save(save_path)

            # Metriche noisy
            psnr_noisy = psnr(clean_np, noisy_np, data_range=1.0)
            ssim_noisy = ssim(clean_np, noisy_np, channel_axis=2, data_range=1.0)

            # Metriche metodo
            psnr_denoised = psnr(clean_np, denoised_np, data_range=1.0)
            ssim_denoised = ssim(clean_np, denoised_np, channel_axis=2, data_range=1.0)

            results.append({
                "method": method_name,
                "filename": str(relative_path),
                "clean_path": str(clean_path),
                "noisy_path": str(noisy_path),
                "denoised_path": str(save_path),
                "psnr_clean_noisy": float(psnr_noisy),
                "ssim_clean_noisy": float(ssim_noisy),
                "psnr_clean_denoised": float(psnr_denoised),
                "ssim_clean_denoised": float(ssim_denoised),
                "psnr_gain": float(psnr_denoised - psnr_noisy),
                "ssim_gain": float(ssim_denoised - ssim_noisy),
            })

        except Exception as e:
            print("Errore su:", noisy_path)
            print(e)

    df = pd.DataFrame(results)

    csv_path = output_dir / csv_name
    df.to_csv(csv_path, index=False)

    print("\nMetodo:", method_name)
    print("CSV salvato in:", csv_path)

    if len(df) > 0:
        print("PSNR medio clean/noisy:", df["psnr_clean_noisy"].mean())
        print("PSNR medio clean/denoised:", df["psnr_clean_denoised"].mean())
        print("Gain PSNR medio:", df["psnr_gain"].mean())

        print("SSIM medio clean/noisy:", df["ssim_clean_noisy"].mean())
        print("SSIM medio clean/denoised:", df["ssim_clean_denoised"].mean())
        print("Gain SSIM medio:", df["ssim_gain"].mean())

    return df

In [13]:
def get_image_paths(input_dir):
    """
    Restituisce tutti i path delle immagini contenute in input_dir,
    includendo eventuali sottocartelle.
    """

    input_dir = Path(input_dir)

    image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]

    image_paths = [
        p for p in input_dir.rglob("*")
        if p.suffix.lower() in image_extensions
    ]

    return sorted(image_paths)

In [12]:
df_kodak_mean3 = evaluate_baseline_dataset(
    method_name="mean_3x3",
    filter_function=lambda img: apply_mean_filter(img, kernel_size=3),
    noisy_dir=kodak_noisy_dir,
    clean_dir=kodak_clean_dir,
    output_dir=output_base_dir / "kodak_lambda30" / "mean_3x3",
    csv_name="kodak_poisson_lambda30_mean_3x3_metrics.csv"
)

mean_3x3: 100%|██████████| 24/24 [00:11<00:00,  2.12it/s]


Metodo: mean_3x3
CSV salvato in: /kaggle/working/poisson_classical_baselines/kodak_lambda30/mean_3x3/kodak_poisson_lambda30_mean_3x3_metrics.csv
PSNR medio clean/noisy: 14.521698844640133
PSNR medio clean/denoised: 22.132983406066746
Gain PSNR medio: 7.611284561426611
SSIM medio clean/noisy: 0.17992970254272223
SSIM medio clean/denoised: 0.42161665111780167
Gain SSIM medio: 0.2416869488855203


## Risultati Kodak->Filtro Media

In [13]:
df_kodak_median3 = evaluate_baseline_dataset(
    method_name="median_3x3",
    filter_function=lambda img: apply_median_filter(img, kernel_size=3),
    noisy_dir=kodak_noisy_dir,
    clean_dir=kodak_clean_dir,
    output_dir=output_base_dir / "kodak_lambda30" / "median_3x3",
    csv_name="kodak_poisson_lambda30_median_3x3_metrics.csv"
)

median_3x3: 100%|██████████| 24/24 [00:12<00:00,  1.97it/s]


Metodo: median_3x3
CSV salvato in: /kaggle/working/poisson_classical_baselines/kodak_lambda30/median_3x3/kodak_poisson_lambda30_median_3x3_metrics.csv
PSNR medio clean/noisy: 14.521698844640133
PSNR medio clean/denoised: 20.381686836926168
Gain PSNR medio: 5.859987992286036
SSIM medio clean/noisy: 0.17992970254272223
SSIM medio clean/denoised: 0.3243009075522423
Gain SSIM medio: 0.14437120563040176


## Risultati Kodak->Filtro Mediano

In [14]:
df_kodak_anscombe_bm3d = evaluate_baseline_dataset(
    method_name="anscombe_bm3d",
    filter_function=lambda img: apply_anscombe_bm3d_rgb(img, sigma=1.0),
    noisy_dir=kodak_noisy_dir,
    clean_dir=kodak_clean_dir,
    output_dir=output_base_dir / "kodak_lambda30" / "anscombe_bm3d",
    csv_name="kodak_poisson_lambda30_anscombe_bm3d_metrics.csv"
)

anscombe_bm3d: 100%|██████████| 24/24 [18:18<00:00, 45.77s/it]


Metodo: anscombe_bm3d
CSV salvato in: /kaggle/working/poisson_classical_baselines/kodak_lambda30/anscombe_bm3d/kodak_poisson_lambda30_anscombe_bm3d_metrics.csv
PSNR medio clean/noisy: 14.521698844640133
PSNR medio clean/denoised: 22.809246011187025
Gain PSNR medio: 8.287547166546892
SSIM medio clean/noisy: 0.17992970254272223
SSIM medio clean/denoised: 0.5862381433447202
Gain SSIM medio: 0.406308442975084


## Risultati Kodak->AnsC

In [20]:
df_bsd500_mean3 = evaluate_baseline_dataset(
    method_name="mean_3x3",
    filter_function=lambda img: apply_mean_filter(img, kernel_size=3),
    noisy_dir=bsd500_test_noisy_dir,
    clean_dir=bsd500_test_clean_dir,
    output_dir=output_base_dir / "bsd500_test_lambda10" / "mean_3x3",
    csv_name="bsd500_test_poisson_lambda10_mean_3x3_metrics.csv"
)

mean_3x3: 100%|██████████| 200/200 [00:26<00:00,  7.65it/s]


Metodo: mean_3x3
CSV salvato in: /kaggle/working/poisson_classical_baselines/bsd500_test_lambda10/mean_3x3/bsd500_test_poisson_lambda10_mean_3x3_metrics.csv
PSNR medio clean/noisy: 17.626273001550697
PSNR medio clean/denoised: 22.80961966301308
Gain PSNR medio: 5.183346661462389
SSIM medio clean/noisy: 0.3185002916306257
SSIM medio clean/denoised: 0.5339582552015781
Gain SSIM medio: 0.21545796371996404


## Risultati BSD500->Filtro Media

In [21]:
df_bsd500_median3 = evaluate_baseline_dataset(
    method_name="median_3x3",
    filter_function=lambda img: apply_median_filter(img, kernel_size=3),
    noisy_dir=bsd500_test_noisy_dir,
    clean_dir=bsd500_test_clean_dir,
    output_dir=output_base_dir / "bsd500_test_lambda10" / "median_3x3",
    csv_name="bsd500_test_poisson_lambda10_median_3x3_metrics.csv"
)

median_3x3: 100%|██████████| 200/200 [00:28<00:00,  7.00it/s]


Metodo: median_3x3
CSV salvato in: /kaggle/working/poisson_classical_baselines/bsd500_test_lambda10/median_3x3/bsd500_test_poisson_lambda10_median_3x3_metrics.csv
PSNR medio clean/noisy: 17.626273001550697
PSNR medio clean/denoised: 22.15967599502586
Gain PSNR medio: 4.533402993475163
SSIM medio clean/noisy: 0.3185002916306257
SSIM medio clean/denoised: 0.48277136191725734
Gain SSIM medio: 0.16427107028663157


## Risultati BSD500->Filtro mediano

In [22]:
df_bsd500_anscombe_bm3d = evaluate_baseline_dataset(
    method_name="anscombe_bm3d",
    filter_function=lambda img: apply_anscombe_bm3d_rgb(img, sigma=1.0),
    noisy_dir=bsd500_test_noisy_dir,
    clean_dir=bsd500_test_clean_dir,
    output_dir=output_base_dir / "bsd500_test_lambda10" / "anscombe_bm3d",
    csv_name="bsd500_test_poisson_lambda10_anscombe_bm3d_metrics.csv"
)

median_3x3: 100%|██████████| 200/200 [00:28<00:00,  6.92it/s]


Metodo: median_3x3
CSV salvato in: /kaggle/working/poisson_classical_baselines/bsd500_test_lambda10/median_3x3/bsd500_test_poisson_lambda10_median_3x3_metrics.csv
PSNR medio clean/noisy: 17.626273001550697
PSNR medio clean/denoised: 22.15967599502586
Gain PSNR medio: 4.533402993475163
SSIM medio clean/noisy: 0.3185002916306257
SSIM medio clean/denoised: 0.48277136191725734
Gain SSIM medio: 0.16427107028663157


## Risultati BSD500->AnsC

In [18]:
df_div2k_mean3 = evaluate_baseline_dataset(
    method_name="mean_3x3",
    filter_function=lambda img: apply_mean_filter(img, kernel_size=3),
    noisy_dir=div2k_valid_noisy_dir,
    clean_dir=div2k_valid_clean_dir,
    output_dir=output_base_dir / "div2k_valid_lambda10" / "mean_3x3",
    csv_name="div2k_valid_poisson_lambda10_mean_3x3_metrics.csv"
)

mean_3x3: 100%|██████████| 100/100 [04:57<00:00,  2.97s/it]


Metodo: mean_3x3
CSV salvato in: /kaggle/working/poisson_classical_baselines/div2k_valid_lambda10/mean_3x3/div2k_valid_poisson_lambda10_mean_3x3_metrics.csv
PSNR medio clean/noisy: 15.034015061331388
PSNR medio clean/denoised: 22.554137040121972
Gain PSNR medio: 7.52012197879059
SSIM medio clean/noisy: 0.2193354954943061
SSIM medio clean/denoised: 0.47947258457541464
Gain SSIM medio: 0.26013708874583247


## Risultati DIV2K->Filtro Media

In [15]:
df_div2k_median3 = evaluate_baseline_dataset(
    method_name="median_3x3",
    filter_function=lambda img: apply_median_filter(img, kernel_size=3),
    noisy_dir=div2k_valid_noisy_dir,
    clean_dir=div2k_valid_clean_dir,
    output_dir=output_base_dir / "div2k_valid_lambda10" / "median_3x3",
    csv_name="div2k_valid_poisson_lambda10_median_3x3_metrics.csv"
)

median_3x3: 100%|██████████| 100/100 [05:41<00:00,  3.42s/it]


Metodo: median_3x3
CSV salvato in: /kaggle/working/poisson_classical_baselines/div2k_valid_lambda10/median_3x3/div2k_valid_poisson_lambda10_median_3x3_metrics.csv
PSNR medio clean/noisy: 15.034015061331388
PSNR medio clean/denoised: 20.662194833267634
Gain PSNR medio: 5.6281797719362485
SSIM medio clean/noisy: 0.2193354954943061
SSIM medio clean/denoised: 0.3602361626923084
Gain SSIM medio: 0.1409006667882204


## Risultati DIV2K->Filtro Mediano

In [17]:
df_div2k_anscombe_bm3d_10 = evaluate_baseline_dataset(
    method_name="anscombe_bm3d",
    filter_function=lambda img: apply_anscombe_bm3d_rgb(img, sigma=1.0),
    noisy_dir=div2k_valid_noisy_dir,
    clean_dir=div2k_valid_clean_dir,
    output_dir=output_base_dir / "div2k_valid_lambda10" / "anscombe_bm3d_10img",
    csv_name="div2k_valid_poisson_lambda10_anscombe_bm3d_10img_metrics.csv",
    max_images=10
)

anscombe_bm3d: 100%|██████████| 10/10 [53:18<00:00, 319.84s/it]


Metodo: anscombe_bm3d
CSV salvato in: /kaggle/working/poisson_classical_baselines/div2k_valid_lambda10/anscombe_bm3d_10img/div2k_valid_poisson_lambda10_anscombe_bm3d_10img_metrics.csv
PSNR medio clean/noisy: 14.600096632681584
PSNR medio clean/denoised: 23.387667961639924
Gain PSNR medio: 8.78757132895834
SSIM medio clean/noisy: 0.19398579075932504
SSIM medio clean/denoised: 0.6321553975343704
Gain SSIM medio: 0.4381696045398712


## Risultati DIV2K->ANSCOMB